In [1]:
import pandas as pd
import random
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import nltk
import numpy as np

In [63]:
# Reading and cleaning the file
df = pd.read_csv('bbc_text.csv', encoding=('ISO-8859-1'))

# Choosing a random sample
random_rows = df[['labels', 'text']].sample(n=1)
data = random_rows.values.tolist()
data

[['entertainment',
  "Howl helps boost Japan's cinemas\r\n\r\nJapan's box office received a 3.8% boost last year, with ticket sales worth 211bn yen (Â£1.08bn).\r\n\r\nThe surge was led by animated movie Howl's Moving Castle, which took 20bn yen (Â£102m) to become the biggest film in Japan in 2004. It is expected to match the 30.7bn yen (Â£157m) record of Hayao Miyazaki's previous film Spirited Away. Japan Motion Picture Producers figures showed that 170 million cinema admissions were made in Japan in 2004. The Last Samurai, starring Tom Cruise, was the biggest foreign movie hit in Japan last year, taking 13.8bn yen (Â£70.7m).\r\n\r\nIt was followed by Harry Potter and the Prisoner of Azkaban, Finding Nemo and The Lord of the Rings: The Return of the King. The second highest-grossing Japanese film was romantic drama Crying Out Love in the Centre of the World, followed by Be With You and Pocket Monsters Advanced Generation. Japanese films accounted for 37.5% of Japan's box office total l

In [65]:
# TFIDF matrix
vectorizer = TfidfVectorizer(stop_words='english')
passage = data[0][1]
sentences = nltk.sent_tokenize(passage)
cleaned_sentences = []
for s in sentences:
    s_clean = s.lower()
    s_clean = re.sub(r'[-/]', ' ', s_clean)
    s_clean = re.sub(r'[^\w\s]', '', s_clean)
    s_clean = re.sub(r'\s+', ' ', s_clean).strip()
    if s_clean: 
        cleaned_sentences.append(s_clean)
matrix = vectorizer.fit_transform(cleaned_sentences)
dense_matrix = matrix.toarray()
size = dense_matrix.shape[0]


In [67]:
# Cosine similarity matrix
G_matrix = np.zeros((size, size))
U_matrix = np.zeros((size, size))
A_matrix = np.zeros((size, size))
a = 0.15
for i in range(size):
    for j in range(size):
        U_matrix[i][j] = 1/size
        if i==j:
            G_matrix[i][j] = 0.0
            continue
        row_i = dense_matrix[i]
        row_j = dense_matrix[j]
        numerator = np.dot(row_i, row_j)
        norm_i = np.linalg.norm(row_i)
        norm_j = np.linalg.norm(row_j)
        if norm_i == 0 or norm_j == 0:
            G_matrix[i][j] = 0.0
            continue
        denuminator = float(norm_i) * float(norm_j)
        G_matrix[i][j] = float(numerator) / denuminator
        
row_sums = G_matrix.sum(axis=1, keepdims=True)
G_matrix = np.divide(G_matrix, row_sums, out=np.zeros_like(G_matrix), where=row_sums != 0)

blank_rows = (row_sums == 0).flatten()
G_matrix[blank_rows] = 1.0 / size

A_matrix = a * U_matrix + (1 - a) * G_matrix
AT_matrix = A_matrix.T

In [69]:
# Limiting distribution
eigenvalues, eigenvectors = np.linalg.eig(AT_matrix)
print('Eigen values:', eigenvalues)

index_of_one = np.argmin(np.abs(eigenvalues - 1.0))
textrank_vector = np.real(eigenvectors[:, index_of_one])
textrank_scores = np.abs(textrank_vector) / np.sum(np.abs(textrank_vector))

print("TextRank Scores:\n", textrank_scores)


Eigen values: [ 1.          0.41008599  0.33054256  0.23516492  0.05181607 -0.47908818
 -0.43300891 -0.35392447 -0.14161696 -0.20806141 -0.26190963]
TextRank Scores:
 [0.08840438 0.1159366  0.06546972 0.06504867 0.12044196 0.02663577
 0.06976561 0.14633648 0.09453734 0.13486456 0.0725589 ]


In [70]:
# Summarizing
sentence_scores = list(zip(sentences, textrank_scores))
sorted_sentences = sorted(sentence_scores, key=lambda x: x[1], reverse=True)
top_sentences = [" ".join(sentence.split()) for sentence, score in sorted_sentences[:4]]
print(" ".join(top_sentences))

Japanese films accounted for 37.5% of Japan's box office total last year, with foreign films taking the remaining 62.5%. The number of Japanese films released rose to 310 in 2004 from 287 the previous year. The Last Samurai, starring Tom Cruise, was the biggest foreign movie hit in Japan last year, taking 13.8bn yen (Â£70.7m). The surge was led by animated movie Howl's Moving Castle, which took 20bn yen (Â£102m) to become the biggest film in Japan in 2004.
